In [1]:
'''
The RNN model is established in PyTorch to perform sentiment analysis on text data.
The model consists of an embedding layer, an RNN layer, and a fully connected layer.
The model outputs a binary sentiment prediction (positive/negative):
The model will be trained on the Financial Sentiment Analysis Dataset. Available at:https://www.kaggle.com/datasets/sbhatti/financial-sentiment-analysis
'''


'\nThe RNN model is established in PyTorch to perform sentiment analysis on text data.\nThe model consists of an embedding layer, an RNN layer, and a fully connected layer.\nThe model outputs a binary sentiment prediction (positive/negative):\nThe model will be trained on the Financial Sentiment Analysis Dataset. Available at:https://www.kaggle.com/datasets/sbhatti/financial-sentiment-analysis\n'

In [5]:
import pandas as pd
import numpy as np

def load_csv_data(csv_file_path):
    """Load CSV data and return text and sentiment columns"""
    # Load the CSV file
    df = pd.read_csv(csv_file_path)
    
    # Extract the two columns (assuming they are named 'text' and 'sentiment')
    texts = df.iloc[:, 0].values  # First column (text)
    sentiments = df.iloc[:, 1].values  # Second column (sentiment)
    
    print(f"Loaded {len(texts)} samples")
    print(f"Sample text: {texts[0]}")
    print(f"Sample sentiment: {sentiments[0]}")
    
    return texts, sentiments

# Usage
csv_file_path = "./finsen/data.csv"  # Update with your CSV path
texts, sentiments = load_csv_data(csv_file_path)

Loaded 5842 samples
Sample text: The GeoSolutions technology will leverage Benefon 's GPS solutions by providing Location Based Search Technology , a Communities Platform , location relevant multimedia content and a new and powerful commercial model .
Sample sentiment: positive


In [6]:
df = pd.read_csv(csv_file_path)
print(df.head())

                                            Sentence Sentiment
0  The GeoSolutions technology will leverage Bene...  positive
1  $ESI on lows, down $1.50 to $2.50 BK a real po...  negative
2  For the last quarter of 2010 , Componenta 's n...  positive
3  According to the Finnish-Russian Chamber of Co...   neutral
4  The Swedish buyout firm has sold its remaining...   neutral


In [8]:
df = df.dropna()

In [9]:
df=df.lower()

AttributeError: 'DataFrame' object has no attribute 'lower'

In [10]:
print(df.head())

                                            Sentence Sentiment
0  The GeoSolutions technology will leverage Bene...  positive
1  $ESI on lows, down $1.50 to $2.50 BK a real po...  negative
2  For the last quarter of 2010 , Componenta 's n...  positive
3  According to the Finnish-Russian Chamber of Co...   neutral
4  The Swedish buyout firm has sold its remaining...   neutral


In [11]:
max_length = df['Sentence'].str.split().str.len().max()
print(f"Longest sentence length: {max_length} words")

Longest sentence length: 81 words


In [12]:
from sklearn.preprocessing import LabelEncoder

# Create label encoder
label_encoder = LabelEncoder()

# Encode the sentiment labels
encoded_labels = label_encoder.fit_transform(df['Sentiment'])

# See the mapping
print("Label mapping:")
for i, label in enumerate(label_encoder.classes_):
    print(f"{label}: {i}")

# Add encoded labels to dataframe
df['encoded_sentiment'] = encoded_labels

Label mapping:
negative: 0
neutral: 1
positive: 2


In [13]:
print(df.head())

                                            Sentence Sentiment  \
0  The GeoSolutions technology will leverage Bene...  positive   
1  $ESI on lows, down $1.50 to $2.50 BK a real po...  negative   
2  For the last quarter of 2010 , Componenta 's n...  positive   
3  According to the Finnish-Russian Chamber of Co...   neutral   
4  The Swedish buyout firm has sold its remaining...   neutral   

   encoded_sentiment  
0                  2  
1                  0  
2                  2  
3                  1  
4                  1  


In [14]:
df.drop(columns=['Sentiment'], inplace=True)
print(df.head())

                                            Sentence  encoded_sentiment
0  The GeoSolutions technology will leverage Bene...                  2
1  $ESI on lows, down $1.50 to $2.50 BK a real po...                  0
2  For the last quarter of 2010 , Componenta 's n...                  2
3  According to the Finnish-Russian Chamber of Co...                  1
4  The Swedish buyout firm has sold its remaining...                  1


In [15]:
from collections import Counter

# Get all words from sentences
all_words = []
for sentence in df['Sentence']:
    words = sentence.split()
    all_words.extend(words)

# Count word frequencies
word_counts = Counter(all_words)

# Create vocabulary (most common words)
vocab_size = 10000  # Set your desired vocab size
most_common = word_counts.most_common(vocab_size - 2)  # -2 for special tokens

# Build word-to-index mapping
word_to_idx = {'<PAD>': 0, '<UNK>': 1}  # Special tokens
for i, (word, count) in enumerate(most_common):
    word_to_idx[word] = i + 2

# Create reverse mapping
idx_to_word = {idx: word for word, idx in word_to_idx.items()}

print(f"Vocabulary size: {len(word_to_idx)}")
print(f"Most common words: {list(word_to_idx.keys())[2:12]}")  # Show first 10 words

Vocabulary size: 10000
Most common words: ['the', '.', ',', 'of', 'in', 'to', 'and', 'a', 'The', 'for']


In [16]:
df['Sentence'] = df['Sentence'].str.lower()

In [17]:
from collections import Counter

# Get all words from sentences
all_words = []
for sentence in df['Sentence']:
    words = sentence.split()
    all_words.extend(words)

# Count word frequencies
word_counts = Counter(all_words)

# Create vocabulary (most common words)
vocab_size = 10000  # Set your desired vocab size
most_common = word_counts.most_common(vocab_size - 2)  # -2 for special tokens

# Build word-to-index mapping
word_to_idx = {'<PAD>': 0, '<UNK>': 1}  # Special tokens
for i, (word, count) in enumerate(most_common):
    word_to_idx[word] = i + 2

# Create reverse mapping
idx_to_word = {idx: word for word, idx in word_to_idx.items()}

print(f"Vocabulary size: {len(word_to_idx)}")
print(f"Most common words: {list(word_to_idx.keys())[2:12]}")  # Show first 10 words


Vocabulary size: 10000
Most common words: ['the', '.', ',', 'of', 'in', 'to', 'and', 'a', 'for', 'eur']


In [18]:
def text_to_sequence(text, word_to_idx, max_length=100):
    """Convert text to sequence of indices"""
    words = text.split()
    sequence = [word_to_idx.get(word, word_to_idx['<UNK>']) for word in words]
    
    # Pad or truncate to max_length
    if len(sequence) < max_length:
        sequence.extend([word_to_idx['<PAD>']] * (max_length - len(sequence)))
    else:
        sequence = sequence[:max_length]
    
    return sequence

# Tokenize all sentences
max_seq_length = 90  # Set your desired sequence length
sequences = []
for sentence in df['Sentence']:
    seq = text_to_sequence(sentence, word_to_idx, max_seq_length)
    sequences.append(seq)

# Convert to numpy array
sequences = np.array(sequences)
print(f"Sequences shape: {sequences.shape}")

Sequences shape: (5842, 90)


In [19]:
print(df.head())

                                            Sentence  encoded_sentiment
0  the geosolutions technology will leverage bene...                  2
1  $esi on lows, down $1.50 to $2.50 bk a real po...                  0
2  for the last quarter of 2010 , componenta 's n...                  2
3  according to the finnish-russian chamber of co...                  1
4  the swedish buyout firm has sold its remaining...                  1


In [21]:
print(sequences[0])

[   2 4085  120   16 3173 1467   13 2248  116   21  774 1250  218 2249
  120    4    9 3174 1168    4 1250 1983 3175  632    8    9   54    8
 4086  529  427    3    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0]


In [22]:
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader

# Split the data
X_train, X_temp, y_train, y_temp = train_test_split(
    sequences, df['encoded_sentiment'], 
    test_size=0.3, random_state=42, stratify=df['encoded_sentiment']
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, 
    test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

# Create PyTorch Dataset
class SentimentDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return {
            'sequence': torch.tensor(self.sequences[idx], dtype=torch.long),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Create datasets
train_dataset = SentimentDataset(X_train, y_train.values)
val_dataset = SentimentDataset(X_val, y_val.values)
test_dataset = SentimentDataset(X_test, y_test.values)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

Train: 4089, Val: 876, Test: 877


In [23]:
import torch.nn as nn

class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, n_layers=1):
        super(SentimentRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.LSTM(embed_dim, hidden_dim, n_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(0.3)
        
    def forward(self, x):
        embedded = self.embedding(x)
        rnn_out, (hidden, _) = self.rnn(embedded)
        output = self.fc(self.dropout(hidden[-1]))
        return output

In [26]:
import torch.optim as optim
model = SentimentRNN(
    vocab_size=len(word_to_idx),
    embed_dim=100,
    hidden_dim=128,
    output_dim=len(df['encoded_sentiment'].unique()),
    n_layers=1
)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

for epoch in range(10):
    running_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()  # Zero the gradients
        outputs = model(batch['sequence'])  # Forward pass
        loss = criterion(outputs, batch['label'])  # Calculate loss
        loss.backward()  # Backpropagation
        optimizer.step()  # Optimize
        running_loss += loss.item()
    print(f'Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}')

Epoch 1, Loss: 0.9897463638335466
Epoch 2, Loss: 0.9876707941293716
Epoch 3, Loss: 0.9857916790060699
Epoch 4, Loss: 0.9861398516222835
Epoch 5, Loss: 0.9828559583984315
Epoch 6, Loss: 0.9816519594751298
Epoch 7, Loss: 0.9817809881642461
Epoch 8, Loss: 0.9842403922230005
Epoch 9, Loss: 0.9851938323117793
Epoch 10, Loss: 0.9830799992196262


In [27]:
# Reload the data properly
df = pd.read_csv(csv_file_path)
df = df.dropna()
df['Sentence'] = df['Sentence'].str.lower()  # Only lowercase the text column

In [28]:
import re
def clean_text(text):
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove punctuation
    return text.strip()

df['Sentence'] = df['Sentence'].apply(clean_text)

In [29]:
print(df.head())
print(f"Unique sentiments: {df['Sentiment'].unique()}")

                                            Sentence Sentiment
0  the geosolutions technology will leverage bene...  positive
1        esi on lows down  to  bk a real possibility  negative
2  for the last quarter of   componenta s net sal...  positive
3  according to the finnishrussian chamber of com...   neutral
4  the swedish buyout firm has sold its remaining...   neutral
Unique sentiments: ['positive' 'negative' 'neutral']


In [30]:
optimizer = optim.Adam(model.parameters(), lr=0.0001)  # Lower LR

In [31]:
from sklearn.preprocessing import LabelEncoder

# Create label encoder
label_encoder = LabelEncoder()

# Encode the sentiment labels
encoded_labels = label_encoder.fit_transform(df['Sentiment'])

# See the mapping
print("Label mapping:")
for i, label in enumerate(label_encoder.classes_):
    print(f"{label}: {i}")

# Add encoded labels to dataframe
df['encoded_sentiment'] = encoded_labels

Label mapping:
negative: 0
neutral: 1
positive: 2


In [32]:
# Complete Data Preprocessing - Fixed Version
import pandas as pd
import numpy as np
import re
from collections import Counter
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader

# 1. Load and clean data properly
csv_file_path = "./finsen/data.csv"
df = pd.read_csv(csv_file_path)
df = df.dropna()

print(f"Original data shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Sample data:")
print(df.head())

# 2. Text cleaning function
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove punctuation and numbers
    text = re.sub(r'\s+', ' ', text)  # Remove extra whitespace
    return text.strip()

# Apply cleaning to text column only
df['Sentence'] = df['Sentence'].apply(clean_text)

# 3. Encode labels
label_encoder = LabelEncoder()
df['encoded_sentiment'] = label_encoder.fit_transform(df['Sentiment'])

print(f"\nLabel mapping:")
for i, label in enumerate(label_encoder.classes_):
    print(f"{label}: {i}")

# 4. Build vocabulary
all_words = []
for sentence in df['Sentence']:
    words = sentence.split()
    all_words.extend(words)

word_counts = Counter(all_words)
vocab_size = 5000  # Reduced vocab size
most_common = word_counts.most_common(vocab_size - 2)

# Build word-to-index mapping
word_to_idx = {'<PAD>': 0, '<UNK>': 1}
for i, (word, count) in enumerate(most_common):
    word_to_idx[word] = i + 2

print(f"\nVocabulary size: {len(word_to_idx)}")

# 5. Get max sentence length for padding
sentence_lengths = df['Sentence'].str.split().str.len()
max_length = min(100, int(sentence_lengths.quantile(0.95)))  # Use 95th percentile, max 100
print(f"Max sequence length: {max_length}")

# 6. Tokenize text
def text_to_sequence(text, word_to_idx, max_length):
    words = text.split()
    sequence = [word_to_idx.get(word, word_to_idx['<UNK>']) for word in words]
    
    # Pad or truncate
    if len(sequence) < max_length:
        sequence.extend([word_to_idx['<PAD>']] * (max_length - len(sequence)))
    else:
        sequence = sequence[:max_length]
    
    return sequence

sequences = []
for sentence in df['Sentence']:
    seq = text_to_sequence(sentence, word_to_idx, max_length)
    sequences.append(seq)

sequences = np.array(sequences)
print(f"Sequences shape: {sequences.shape}")

# 7. Split data
X_train, X_temp, y_train, y_temp = train_test_split(
    sequences, df['encoded_sentiment'], 
    test_size=0.3, random_state=42, stratify=df['encoded_sentiment']
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, 
    test_size=0.5, random_state=42, stratify=y_temp
)

print(f"\nData splits - Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

# 8. Create PyTorch Dataset
class SentimentDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return {
            'sequence': torch.tensor(self.sequences[idx], dtype=torch.long),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# 9. Create datasets and dataloaders
train_dataset = SentimentDataset(X_train, y_train.values)
val_dataset = SentimentDataset(X_val, y_val.values)
test_dataset = SentimentDataset(X_test, y_test.values)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"\nDataLoaders created:")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

# 10. Print final info for model creation
print(f"\nModel parameters:")
print(f"Vocab size: {len(word_to_idx)}")
print(f"Sequence length: {max_length}")
print(f"Number of classes: {len(label_encoder.classes_)}")

print("\n✅ Data preprocessing completed successfully!")

Original data shape: (5842, 2)
Columns: ['Sentence', 'Sentiment']
Sample data:
                                            Sentence Sentiment
0  The GeoSolutions technology will leverage Bene...  positive
1  $ESI on lows, down $1.50 to $2.50 BK a real po...  negative
2  For the last quarter of 2010 , Componenta 's n...  positive
3  According to the Finnish-Russian Chamber of Co...   neutral
4  The Swedish buyout firm has sold its remaining...   neutral

Label mapping:
negative: 0
neutral: 1
positive: 2

Vocabulary size: 5000
Max sequence length: 34
Sequences shape: (5842, 34)

Data splits - Train: 4089, Val: 876, Test: 877

DataLoaders created:
Train batches: 128
Val batches: 28
Test batches: 28

Model parameters:
Vocab size: 5000
Sequence length: 34
Number of classes: 3

✅ Data preprocessing completed successfully!


In [ ]:
# Model Definition and Training - Complete Pipeline
import torch
import torch.nn as nn
import torch.optim as optim

# Define the RNN Model
class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, n_layers=1):
        super(SentimentRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.LSTM(embed_dim, hidden_dim, n_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(0.3)
        
    def forward(self, x):
        embedded = self.embedding(x)
        rnn_out, (hidden, _) = self.rnn(embedded)
        output = self.fc(self.dropout(hidden[-1]))
        return output

# Initialize the model
model = SentimentRNN(
    vocab_size=len(word_to_idx),
    embed_dim=100,
    hidden_dim=128,
    output_dim=len(label_encoder.classes_),
    n_layers=1
)

# Define optimizer and loss function
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

print(f"Model initialized with:")
print(f"- Vocab size: {len(word_to_idx)}")
print(f"- Embedding dim: 100")
print(f"- Hidden dim: 128")
print(f"- Output classes: {len(label_encoder.classes_)}")
print(f"- Learning rate: 0.001")
print("-" * 50)

# Training loop for 100 epochs
num_epochs = 100
train_losses = []
val_losses = []

for epoch in range(num_epochs):
    # Training phase
    model.train()
    running_train_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for batch in train_loader:
        # Zero gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(batch['sequence'])
        loss = criterion(outputs, batch['label'])
        
        # Backward pass and optimization
        loss.backward()
        optimizer.step()
        
        # Statistics
        running_train_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total_train += batch['label'].size(0)
        correct_train += (predicted == batch['label']).sum().item()
    
    # Calculate average training loss and accuracy
    avg_train_loss = running_train_loss / len(train_loader)
    train_accuracy = 100 * correct_train / total_train
    train_losses.append(avg_train_loss)
    
    # Validation phase
    model.eval()
    running_val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad():
        for batch in val_loader:
            outputs = model(batch['sequence'])
            loss = criterion(outputs, batch['label'])
            
            running_val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_val += batch['label'].size(0)
            correct_val += (predicted == batch['label']).sum().item()
    
    # Calculate average validation loss and accuracy
    avg_val_loss = running_val_loss / len(val_loader)
    val_accuracy = 100 * correct_val / total_val
    val_losses.append(avg_val_loss)
    
    # Print epoch results
    print(f'Epoch [{epoch+1}/{num_epochs}]:')
    print(f'  Train Loss: {avg_train_loss:.4f}, Train Acc: {train_accuracy:.2f}%')
    print(f'  Val Loss: {avg_val_loss:.4f}, Val Acc: {val_accuracy:.2f}%')
    print('-' * 50)

print("🎉 Training completed!")
print(f"Final Training Accuracy: {train_accuracy:.2f}%")
print(f"Final Validation Accuracy: {val_accuracy:.2f}%")

# Test the model on test set
model.eval()
correct_test = 0
total_test = 0
test_predictions = []
test_labels = []

with torch.no_grad():
    for batch in test_loader:
        outputs = model(batch['sequence'])
        _, predicted = torch.max(outputs.data, 1)
        total_test += batch['label'].size(0)
        correct_test += (predicted == batch['label']).sum().item()
        
        test_predictions.extend(predicted.cpu().numpy())
        test_labels.extend(batch['label'].cpu().numpy())

test_accuracy = 100 * correct_test / total_test
print(f"\n📊 Final Test Accuracy: {test_accuracy:.2f}%")

# Save the model
torch.save(model.state_dict(), 'sentiment_rnn_model.pth')
print("\n💾 Model saved as 'sentiment_rnn_model.pth'")

Model initialized with:
- Vocab size: 5000
- Embedding dim: 100
- Hidden dim: 128
- Output classes: 3
- Learning rate: 0.001
--------------------------------------------------
Epoch [1/100]:
  Train Loss: 0.9920, Train Acc: 53.02%
  Val Loss: 0.9829, Val Acc: 53.54%
--------------------------------------------------
Epoch [1/100]:
  Train Loss: 0.9920, Train Acc: 53.02%
  Val Loss: 0.9829, Val Acc: 53.54%
--------------------------------------------------
Epoch [2/100]:
  Train Loss: 0.9666, Train Acc: 54.32%
  Val Loss: 0.9370, Val Acc: 57.76%
--------------------------------------------------
Epoch [2/100]:
  Train Loss: 0.9666, Train Acc: 54.32%
  Val Loss: 0.9370, Val Acc: 57.76%
--------------------------------------------------
Epoch [3/100]:
  Train Loss: 0.8890, Train Acc: 59.75%
  Val Loss: 0.8932, Val Acc: 62.79%
--------------------------------------------------
Epoch [3/100]:
  Train Loss: 0.8890, Train Acc: 59.75%
  Val Loss: 0.8932, Val Acc: 62.79%
-----------------------